# Step 23 — prepare GSE232381: counts to log counts per million

**Data type: bulk_RNA_seq** (GSE232381). **Reads:** `data/GSE232381/`, NCBI gene_info.
**Writes:** `step23_GSE232381.rds`.

RNA sequencing counts are not on the array's scale, so they are normalised the RNA sequencing way,
inside this study only:

1. keep genes with enough counts to measure (`edgeR::filterByExpr`);
2. correct for library size and composition with TMM (trimmed mean of M-values);
3. convert to log2 counts per million with voom (limma).

The result is a log2 scale like the array's, but **its values are never placed next to array
values**. Steps 26–31 compare studies only in standardised units and scale-free scores.

In [1]:
source("../src/paths.R")
source("../src/platforms.R")
suppressMessages({library(GEOquery); library(Biobase)})
cnt <- read.delim(raw("GSE232381", "GSE232381_raw_counts_GRCh38.p13_NCBI.tsv.gz"), check.names = FALSE)
counts <- as.matrix(cnt[, -1]); rownames(counts) <- cnt$GeneID
stopifnot(all(counts == round(counts)), all(counts >= 0))       # raw integer counts
dim(counts)

[1] 39376    16

## Sample labels

The only clinical field in GEO is active against inactive lupus nephritis. **GEO records no age, sex
or SLEDAI.** The parent cohort mixes patients with pediatric-onset lupus who are now adults with
patients still under 18, so whether these 16 are children cannot be confirmed from GEO. We keep only
the GEO accession and the activity label. The free-text sample descriptions are not carried forward.

In [2]:
pd <- pData(suppressMessages(getGEO(filename = raw("GSE232381", "GSE232381_series_matrix.txt.gz"), getGPL = FALSE)))
meta <- data.frame(sample = rownames(pd),
                   ln_activity = factor(sub(" LN$", "", pd[["treatment:ch1"]]), levels = c("inactive", "active")),
                   row.names = rownames(pd))
meta <- meta[colnames(counts), , drop = FALSE]
table(meta$ln_activity)


inactive   active 
       6       10 

## Entrez identifiers to gene symbols

In [3]:
gi <- read_gene_info()
sym <- gi$Symbol[match(rownames(counts), gi$GeneID)]
c(genes = nrow(counts), with_symbol = sum(!is.na(sym)))
counts <- collapse_max_mean(counts, sym)

genes with_symbol 
      39376       37663

## Normalise

In [4]:
norm <- counts_to_logcpm(counts, group = meta$ln_activity)
E <- norm$E
c(genes_in = norm$genes_in, genes_measured = norm$genes_kept)
round(quantile(E), 2)
read_gene_set("ifn-type1-6.txt") %in% rownames(E)

genes_in genes_measured 
         37662          14692

0%   25%   50%   75%  100% 
-6.55  0.92  2.76  4.34 19.26

[1] TRUE TRUE TRUE TRUE TRUE TRUE

In [5]:
saveRDS(list(E = E, meta = meta, data_type = "bulk_RNA_seq", tissue = "peripheral blood cells"),
        art("step23_GSE232381.rds"))
cat("wrote", art("step23_GSE232381.rds"), "\n")

wrote /Users/adeslatt/Scitechcon Dropbox/Anne DeslattesMays/projects/endotypes-transcriptomics/data/run_artifacts/step23_GSE232381.rds 


## Findings

16 samples, 10 active and 6 inactive lupus nephritis. 14,692 of 37,662 genes have enough counts to
measure, and all six interferon-score genes are among them. Nothing here says whether these patients
are children.